In [ ]:
# 02_contact_probability_and_rsweep.ipynb
# Pairwise contact probability under the seeding/counting convention.
# Confirms the leading constant q_pred = pi*r*Lbar^2*|C|/|S|^2 as r -> 0,
# together with the finite-r correction factor 1 + 8r/Lbar (Appendix B).
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 10, "axes.edgecolor": "0.3"})
GRAY = "0.35"
FIG_DIR, TAB_DIR = "../results/figures", "../results/tables"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)


def savefig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), bbox_inches="tight")


rng = np.random.default_rng(20261006)

ell = np.array([0.2, 0.4])
p = np.array([0.5, 0.5])
lmax = ell.max()
Lbar = float((p * ell).sum())
Cvol, Svol = 1.0, (1.0 + lmax) ** 3


def seg_seg_dist_mid(P1, P2, Q1, Q2):
    """Minimum distance between segment [P1,P2] and [Q1,Q2], and the midpoint
    of the closest pair of points (clamped Lumelsky scheme), vectorized."""
    d1, d2, r = P2 - P1, Q2 - Q1, P1 - Q1
    a = (d1 * d1).sum(1); e = (d2 * d2).sum(1); f = (d2 * r).sum(1)
    b = (d1 * d2).sum(1); c = (d1 * r).sum(1)
    den = a * e - b * b
    s = np.where(den > 1e-12, np.clip((b * f - c * e) / den, 0, 1), 0.0)
    t = np.clip((b * s + f) / e, 0, 1)
    s = np.clip((b * t - c) / a, 0, 1)
    cpA, cpB = P1 + s[:, None] * d1, Q1 + t[:, None] * d2
    return np.linalg.norm(cpA - cpB, axis=1), 0.5 * (cpA + cpB)


def sample_fibers(n, rng):
    L = rng.choice(ell, size=n, p=p)
    X = rng.uniform(-lmax / 2, 1 + lmax / 2, size=(n, 3))
    v = rng.normal(size=(n, 3)); v /= np.linalg.norm(v, axis=1, keepdims=True)
    return X - 0.5 * L[:, None] * v, X + 0.5 * L[:, None] * v


def estimate_q(twor, target_hits, rng, batch=2_000_000):
    """Monte Carlo pairwise contact probability; a contact requires axis
    distance <= 2r and contact point inside the cube."""
    q_pred = np.pi * (twor / 2) * Lbar ** 2 * Cvol / Svol ** 2
    n_pairs = int(max(target_hits / q_pred, 5e5))
    hits, done = 0, 0
    while done < n_pairs:
        m = min(batch, n_pairs - done)
        A1, A2 = sample_fibers(m, rng)
        B1, B2 = sample_fibers(m, rng)
        dist, mid = seg_seg_dist_mid(A1, A2, B1, B2)
        inC = np.all((mid >= 0) & (mid <= 1), axis=1)
        hits += int(np.sum((dist <= twor) & inC))
        done += m
    return q_pred, hits / n_pairs, n_pairs


rows = []
for twor in [0.020, 0.010, 0.004, 0.002]:          # r = 0.010, 0.005, 0.002, 0.001
    q_pred, q_obs, npairs = estimate_q(twor, target_hits=4000, rng=rng)
    rows.append({"r": twor / 2, "q_pred": q_pred, "q_obs": q_obs,
                 "ratio": q_obs / q_pred, "n_pairs": npairs})
table = pd.DataFrame(rows)
print(table.to_string(index=False))
table.to_latex(os.path.join(TAB_DIR, "table_rsweep.tex"), index=False, float_format="%.3e")

# --- Figure: observed/predicted ratio vs r, with the 1 + 8r/Lbar correction ---
r_grid = np.linspace(0, table["r"].max() * 1.05, 100)
fig, ax = plt.subplots(figsize=(4.6, 3.2))
ax.plot(r_grid, 1 + 8 * r_grid / Lbar, color="black", lw=1.3,
        label=r"$1 + 8r/\overline{L}$")
ax.scatter(table["r"], table["ratio"], s=44, facecolor=GRAY, edgecolor="black",
           zorder=3, label=r"$q_{\mathrm{obs}}/q_{\mathrm{pred}}$")
ax.axhline(1.0, color="0.6", ls=":", lw=1)
ax.set_xlabel(r"fiber radius $r$")
ax.set_ylabel(r"$q_{\mathrm{obs}}/q_{\mathrm{pred}}$")
ax.legend(frameon=False)
savefig(fig, "fig_rsweep")
plt.show()
